In [ ]:
# Phase 2: Unified Preprocessing (Live + Archive)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

# ---------------------------------------------------------
# 1. LOAD DATASETS
# ---------------------------------------------------------
print("📂 Loading Datasets...")

# Load Live Data (Phase 1)
try:
    df_live = pd.read_csv('data/raw/master_dataset_real.csv')
    print(f"   > Live Data (Recent): {len(df_live)} rows")
except FileNotFoundError:
    print("   ⚠️ Live Data not found. Skipping.")
    df_live = pd.DataFrame()

# Load Archive Data (Phase 1B)
try:
    df_archive = pd.read_csv('data/raw/unified_archive_data.csv')
    # Keep only the columns that match Live Data
    common_cols = ['latitude', 'longitude', 'acq_date', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'fire_detected']
    # Filter to ensure we only have these columns (ignore extra noise)
    df_archive = df_archive[common_cols] if not df_archive.empty else pd.DataFrame()
    print(f"   > Archive Data (Historical): {len(df_archive)} rows")
except FileNotFoundError:
    print("   ⚠️ Archive Data not found. Skipping.")
    df_archive = pd.DataFrame()

# ---------------------------------------------------------
# 2. MERGE & UNIFY
# ---------------------------------------------------------
if df_live.empty and df_archive.empty:
    raise ValueError("❌ CRITICAL ERROR: No data found! Run Phase 1 or 1B first.")

# Concatenate both datasets
df = pd.concat([df_live, df_archive], ignore_index=True)
print(f"🔗 Total Combined Data: {len(df)} rows")

# ---------------------------------------------------------
# 3. FEATURE ENGINEERING
# ---------------------------------------------------------
print("⚙️ Engineering Features...")

# Parse Dates
df['acq_date'] = pd.to_datetime(df['acq_date'])

# Extract Time Features (AI understands numbers, not dates)
df['month'] = df['acq_date'].dt.month
# We don't use 'day' or 'year' as they might overfit to specific events. Month captures Seasonality.

# Define the "Golden Features" (The Physics of Fire)
features = ['latitude', 'longitude', 'month', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
target = 'fire_detected'

# Create X (Inputs) and y (Output)
X = df[features]
y = df[target]

# ---------------------------------------------------------
# 4. CLEANING (IMPUTATION)
# ---------------------------------------------------------
# If API failed for some rows, fill gaps with the average (Mean)
print("🧹 Cleaning Missing Values...")
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=features)

# ---------------------------------------------------------
# 5. DATA IMBALANCE CHECK
# ---------------------------------------------------------
# Check if we have too many Fires or too many Safe points
fire_count = y.value_counts().get(1, 0)
safe_count = y.value_counts().get(0, 0)
print(f"📊 Distribution: {fire_count} Fires | {safe_count} Safe")

# ---------------------------------------------------------
# 6. SPLIT & SCALE
# ---------------------------------------------------------
print("✂️ Splitting Data (80% Train, 20% Test)...")
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, stratify=y, random_state=42)

print("⚖️ Scaling Data (Normalization)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 7. SAVE THE PROCESSED BRAIN
# ---------------------------------------------------------
print("💾 Saving Processed Files for Phase 3...")
np.save('data/processed/X_train.npy', X_train_scaled)
np.save('data/processed/X_test.npy', X_test_scaled)
np.save('data/processed/y_train.npy', y_train)
np.save('data/processed/y_test.npy', y_test)

# Save the Tools (We need these for the Final App)
joblib.dump(scaler, 'data/processed/scaler.pkl')
joblib.dump(imputer, 'data/processed/imputer.pkl')
joblib.dump(features, 'data/processed/feature_names.pkl')

print("✅ Phase 2 Complete. System is ready for Training.")

# ---------------------------------------------------------
# 8. VISUALIZATION (THE HEATMAP)
# ---------------------------------------------------------
print("\n🎨 Generating Physics Heatmap...")
plt.figure(figsize=(10, 8))

# Combine X and y temporarily just for the plot
plot_data = X_imputed.copy()
plot_data['fire_detected'] = y.values

# Calculate Correlation
corr = plot_data.corr()

# Plot
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Physics of Fire: Correlation Matrix (Live + Archive)")
plt.show()

print("Interpreting the Heatmap:")
print("1. Look at the 'fire_detected' row.")
print("2. RED squares = Positive Risk (e.g., Temperature).")
print("3. BLUE squares = Negative Risk (e.g., Humidity).")